# Network Graph



#### Discriptions of variables
- supply: actual generation output at the bus in MW (Act.P[MW]) supplyb​=g∈generators at bus b ∑​ Act.Pg​
- demand: the total active power consumption in MW assigned to a bus, based on the aggregated load entries in the Load sheet. demandb​=l∈loads at bus b ∑ ​Act.P
- p_min: minimum generation level at the bus in MW
- p_max: maximum generation capacity at the bus in MW
- source: generation technology type(s) connected to the bus (wind, sollar...)
- node_role: whether the bus functions as generator, load, both, or neither
- capacity≈ squareroot(3)⋅VkV​⋅IkA​

### Use the object without haveing to run the jupyter: 

In [23]:
import pickle

with open("network_graph.pkl", "rb") as f:
    G = pickle.load(f)

### Loading pakages

In [10]:
import pandas as pd
import numpy as np
import networkx as nx

### Loading raw data 

In [11]:
file_path = "publicdataexportv131450706334_with_lon_lat.xlsx"

def to_num(series):
    return pd.to_numeric(
        series.astype(str).str.replace(",", ".", regex=False),
        errors="coerce"
    )


# Loading sheets
bus = pd.read_excel(file_path, sheet_name="Bus", header=3)
line = pd.read_excel(file_path, sheet_name="Line", header=3)
gen = pd.read_excel(file_path, sheet_name="Generator", header=3)
load = pd.read_excel(file_path, sheet_name="Load", header=3)
hvdc = pd.read_excel(file_path, sheet_name="HVDC", header=3)

for df in [bus, line, gen, load, hvdc]:
    df.columns = [str(c).strip() for c in df.columns]


### Cleaning "bus" (first sheet) and creating nodes

In [12]:
bus["Bus Index"] = pd.to_numeric(bus["Bus Index"], errors="coerce")
bus["lon"] = pd.to_numeric(bus["lon"], errors="coerce")
bus["lat"] = pd.to_numeric(bus["lat"], errors="coerce")
bus["Voltage base[kV]"] = to_num(bus["Voltage base[kV]"])

nodes = bus[[
    "Bus Index",
    "Bus Name",
    "Station Full Name",
    "Location Name",
    "Voltage base[kV]",
    "lon",
    "lat"
]].copy()


nodes = nodes.rename(columns={
    "Bus Index": "bus_index",
    "Bus Name": "bus_name",
    "Station Full Name": "station_full_name",
    "Location Name": "area_code",
    "Voltage base[kV]": "voltage_kv"
})

nodes = nodes.dropna(subset=["bus_index"]).drop_duplicates(subset=["bus_index"])
nodes["bus_index"] = nodes["bus_index"].astype(int)

# name = station_full_name hvis den findes, ellers bus_name
nodes["name"] = nodes["station_full_name"].fillna("").astype(str).str.strip()
nodes.loc[nodes["name"] == "", "name"] = nodes["bus_name"]

#### We aggregate the generator sheet from unit level to bus level by grouping all generators connected to the same bus and summing their actual output, minimum output, and maximum capacity, so that each graph node can carry total generation attributes in MW.

In [13]:
gen["Bus Index"] = pd.to_numeric(gen["Bus Index"], errors="coerce")
gen["Pmin[MW]"] = to_num(gen["Pmin[MW]"])
gen["Pmax[MW]"] = to_num(gen["Pmax[MW]"])
gen["Act.P[MW]"] = to_num(gen["Act.P[MW]"])

# --- classify source type from Generator Name ---
def classify_generator_source(name):
    if pd.isna(name):
        return "unknown"
    name = str(name).lower()

    if "windoff" in name:
        return "wind_offshore"
    elif "windon" in name:
        return "wind_onshore"
    elif "solar" in name:
        return "solar"
    elif "gas" in name:
        return "gas"
    elif "hydro" in name:
        return "hydro"
    elif "other" in name:
        return "other"
    else:
        return "unknown"

gen["gen_source"] = gen["Generator Name"].apply(classify_generator_source)

# --- aggregate MW values by bus ---
gen_agg = (
    gen.groupby("Bus Index", dropna=False)
    .agg(
        supply=("Act.P[MW]", "sum"),
        p_min=("Pmin[MW]", "sum"),
        p_max=("Pmax[MW]", "sum")
    )
    .reset_index()
    .rename(columns={"Bus Index": "bus_index"})
)

gen_agg["bus_index"] = pd.to_numeric(gen_agg["bus_index"], errors="coerce")
gen_agg = gen_agg.dropna(subset=["bus_index"])
gen_agg["bus_index"] = gen_agg["bus_index"].astype(int)

# --- aggregate source types by bus ---
source_agg = (
    gen.groupby("Bus Index")["gen_source"]
    .apply(lambda x: ", ".join(sorted(set(v for v in x.dropna() if v != "unknown"))))
    .reset_index()
    .rename(columns={
        "Bus Index": "bus_index",
        "gen_source": "source"
    })
)

source_agg["bus_index"] = pd.to_numeric(source_agg["bus_index"], errors="coerce")
source_agg = source_agg.dropna(subset=["bus_index"])
source_agg["bus_index"] = source_agg["bus_index"].astype(int)

# hvis en bus kun havde ukendte generatornavne, så bliver source tom
source_agg["source"] = source_agg["source"].replace("", np.nan)

### Aggregating the load sheet to bus level by grouping all load entries assigned to the same bus, summing their actual active power values, and storing the result as positive MW demand for each graph node.

In [14]:
# Assumption: Load Index corresponds to Bus Index
load["Load Index"] = pd.to_numeric(load["Load Index"], errors="coerce")
load["Act.P[MW]"] = to_num(load["Act.P[MW]"])

load_agg = (
    load.groupby("Load Index", dropna=False)
    .agg(demand=("Act.P[MW]", "sum"))
    .reset_index()
    .rename(columns={"Load Index": "bus_index"})
)

load_agg["bus_index"] = pd.to_numeric(load_agg["bus_index"], errors="coerce")
load_agg = load_agg.dropna(subset=["bus_index"])
load_agg["bus_index"] = load_agg["bus_index"].astype(int)

# Make demand positive for consumption
# If a load entry is negative (e.g. foreign/border convention), take abs
load_agg["demand"] = load_agg["demand"].abs()

### Merging nodes attributes
#### We merge the aggregated generation and load summaries onto the bus-level node table, replace missing values with zero for buses without attached generation or demand, classify each bus by functional role, and retain the final set of node attributes used in the graph model.

In [15]:
nodes = nodes.merge(gen_agg, on="bus_index", how="left")
nodes = nodes.merge(load_agg, on="bus_index", how="left")
nodes = nodes.merge(source_agg, on="bus_index", how="left")


for col in ["supply", "p_min", "p_max", "demand"]:
    nodes[col] = nodes[col].fillna(0.0)

def classify_node_role(row):
    has_gen = row["p_max"] > 0
    has_load = row["demand"] > 0
    
    if has_gen and has_load:
        return "generator+load"
    elif has_gen:
        return "generator"
    elif has_load:
        return "load"
    else:
        return "none"

nodes["node_role"] = nodes.apply(classify_node_role, axis=1)


nodes_final = nodes[[
    "bus_index",
    "name",
    "supply",
    "demand",
    "p_min",
    "p_max",
    "node_role",
    "source",
    "bus_name",
    "station_full_name",
    "area_code",
    "voltage_kv",
    "lat",
    "lon"
]].copy()

#### Making edges

In [16]:
line["Node 1"] = pd.to_numeric(line["Node 1"], errors="coerce")
line["Node 2"] = pd.to_numeric(line["Node 2"], errors="coerce")
line["Nominal Current[kA]"] = to_num(line["Nominal Current[kA]"])
line["Nominal Voltage[kV]"] = to_num(line["Nominal Voltage[kV]"])

edges = line.rename(columns={
    "Node 1": "node1",
    "Node 2": "node2",
    "Line name": "name"
}).copy()

edges = edges.dropna(subset=["node1", "node2"]).copy()
edges["node1"] = edges["node1"].astype(int)
edges["node2"] = edges["node2"].astype(int)

# Approximate edge capacity in MW from line rating
# capacity ≈ sqrt(3) * V[kV] * I[kA]
edges["capacity"] = np.sqrt(3) * edges["Nominal Voltage[kV]"] * edges["Nominal Current[kA]"]

edges_final = edges[[
    "node1",
    "node2",
    "name",
    "capacity",
    "Area Name",
    "Line type",
    "Nominal Voltage[kV]",
    "Nominal Current[kA]",
    "R1[Ohm]",
    "X1[Ohm]",
    "Length[km]"
]].copy()

edges_final = edges_final.rename(columns={
    "Area Name": "area_code",
    "Line type": "line_type",
    "Nominal Voltage[kV]": "voltage_kv",
    "Nominal Current[kA]": "nominal_current_ka",
    "R1[Ohm]": "r_ohm",
    "X1[Ohm]": "x_ohm",
    "Length[km]": "length_km"
})



#### Saving clean tables

In [17]:
out_file = "danish_grid_graph_ready.xlsx"
with pd.ExcelWriter(out_file, engine="openpyxl") as writer:
    nodes_final.to_excel(writer, sheet_name="nodes", index=False)
    edges_final.to_excel(writer, sheet_name="edges", index=False)

print(f"Saved graph-ready dataset to {out_file}")
print("Nodes:", nodes_final.shape)
print("Edges:", edges_final.shape)



Saved graph-ready dataset to danish_grid_graph_ready.xlsx
Nodes: (300, 14)
Edges: (358, 11)


#### Building the network graph


In [18]:
G = nx.Graph()

# add nodes
for _, row in nodes_final.iterrows():
    G.add_node(
        int(row["bus_index"]),
        name=row["name"],
        supply=float(row["supply"]),
        demand=float(row["demand"]),
        p_min=float(row["p_min"]),
        p_max=float(row["p_max"]),
        source=row["source"]
    )

# add edges
for _, row in edges_final.iterrows():
    G.add_edge(
        int(row["node1"]),
        int(row["node2"]),
        name=row["name"],
        capacity=float(row["capacity"]) if pd.notna(row["capacity"]) else 0.0
    )

print("Graph created.")
print("Number of nodes:", G.number_of_nodes())
print("Number of edges:", G.number_of_edges())

# Optional sanity check
sample_node = list(G.nodes(data=True))[:5]
sample_edge = list(G.edges(data=True))[:5]

print("\nSample nodes:")
for x in sample_node:
    print(x)

print("\nSample edges:")
for x in sample_edge:
    print(x)


Graph created.
Number of nodes: 300
Number of edges: 322

Sample nodes:
(76, {'name': 'Bjæverskov', 'supply': 0.0, 'demand': 0.0, 'p_min': 0.0, 'p_max': 0.0, 'source': nan})
(192, {'name': '150 KV STATION ABILDSKOV', 'supply': 27.47, 'demand': 33.9, 'p_min': 0.0, 'p_max': 75.69, 'source': 'gas, hydro, solar, wind_onshore'})
(194, {'name': '150 KV STATION ÅDALEN', 'supply': 3.7899999999999996, 'demand': 44.7, 'p_min': 0.0, 'p_max': 9.94, 'source': 'gas, solar, wind_onshore'})
(195, {'name': 'Anholt Havmøllepark', 'supply': 52.68, 'demand': 0.0, 'p_min': 0.0, 'p_max': 400.0, 'source': 'wind_offshore'})
(53, {'name': '132 KV STATION ALLERØDGÅRD', 'supply': 21.21, 'demand': 58.0, 'p_min': 0.0, 'p_max': 75.0, 'source': 'gas, solar, wind_onshore'})

Sample edges:
(192, 284, {'name': 'ABS_150_OVE', 'capacity': 220.05705510162588})
(192, 299, {'name': 'ABS_150_SVB_eq_ElmLne', 'capacity': 282.9304994163761})
(194, 216, {'name': 'ADL_150_FER_eq_ElmLne', 'capacity': 265.78319642144425})
(194, 245

In [19]:
print(G.nodes[2])

{'name': 'SÅN_400_S1', 'supply': 0.0, 'demand': 500.0, 'p_min': 0.0, 'p_max': 0.0, 'source': nan}


In [21]:
import pickle

with open("network_graph.pkl", "wb") as f:
    pickle.dump(G, f)